# Order Fulfillment Bottleneck Analysis

## Project Overview
Analyze order fulfillment steps to identify which stages delay completion and how that varies by order type or warehouse. This is a descriptive analytics project.

## Learning Objectives
- Decompose fulfillment into stages and measure each
- Identify bottleneck stages by warehouse and order type
- Compare fulfillment speed across operational dimensions
- Detect patterns in stage-level delays

## Problem Statement
Customers complain about slow order fulfillment. Operations needs to know which stage — picking, packing, QC, or shipping — is the bottleneck, and whether it varies by warehouse or order type.

## Why This Project Matters
Fulfillment speed is a competitive advantage. Stage-level bottleneck identification enables targeted investment rather than blanket capacity increases.

## Dataset Overview
Synthetic fulfillment data: ~6,000 orders across 4 warehouses with timestamps for 4 fulfillment stages.

## Dataset Source and License Notes
- Synthetic data for educational purposes
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 6000
warehouses = np.random.choice(['WH-North', 'WH-South', 'WH-East', 'WH-West'], n, p=[0.30, 0.25, 0.25, 0.20])
order_types = np.random.choice(['Single-Item', 'Multi-Item', 'Bulk', 'Fragile'], n, p=[0.35, 0.30, 0.20, 0.15])

dates = pd.date_range('2023-01-01', '2024-12-31', freq='D')
order_dates = np.array([pd.Timestamp(d) for d in np.random.choice(dates, n)])

# Stage durations (minutes) with warehouse and type effects
wh_pick = {'WH-North': 25, 'WH-South': 35, 'WH-East': 28, 'WH-West': 40}
type_pack = {'Single-Item': 10, 'Multi-Item': 25, 'Bulk': 45, 'Fragile': 35}

pick_min = np.array([max(5, int(np.random.normal(wh_pick[w], 8))) for w in warehouses])
pack_min = np.array([max(5, int(np.random.normal(type_pack[t], 7))) for t in order_types])
qc_min = np.array([max(3, int(np.random.exponential(12))) for _ in range(n)])
ship_min = np.array([max(5, int(np.random.normal(20, 8))) for _ in range(n)])
total_min = pick_min + pack_min + qc_min + ship_min

# SLA: 120 min for standard
sla_target = 120
on_time = (total_min <= sla_target).astype(int)

df = pd.DataFrame({
    'OrderID': [f'ORD{i:06d}' for i in range(n)],
    'OrderDate': order_dates, 'Warehouse': warehouses, 'OrderType': order_types,
    'PickMinutes': pick_min, 'PackMinutes': pack_min,
    'QCMinutes': qc_min, 'ShipMinutes': ship_min,
    'TotalMinutes': total_min, 'OnTimeSLA': on_time,
})
df['YearMonth'] = df['OrderDate'].dt.to_period('M')

print(f'Dataset shape: {df.shape}')
print(f'Avg fulfillment: {df["TotalMinutes"].mean():.0f} min')
print(f'SLA compliance: {df["OnTimeSLA"].mean():.1%}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Date range: {df["OrderDate"].min().date()} to {df["OrderDate"].max().date()}')
print(f'\nWarehouse distribution:\n{df["Warehouse"].value_counts()}')
print(f'\nOrder type distribution:\n{df["OrderType"].value_counts()}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

stages = ['PickMinutes', 'PackMinutes', 'QCMinutes', 'ShipMinutes']
df[stages].mean().plot.bar(ax=axes[0,0], edgecolor='black', color=['coral','steelblue','green','purple'])
axes[0,0].set_title('Avg Duration by Stage (minutes)')
axes[0,0].tick_params(axis='x', rotation=45)

df.groupby('Warehouse')['TotalMinutes'].mean().sort_values().plot.barh(ax=axes[0,1], edgecolor='black', color='teal')
axes[0,1].set_title('Avg Total Fulfillment by Warehouse')

df.groupby('OrderType')['TotalMinutes'].mean().sort_values().plot.barh(ax=axes[1,0], edgecolor='black', color='coral')
axes[1,0].set_title('Avg Total Fulfillment by Order Type')

df.groupby('Warehouse')['OnTimeSLA'].mean().sort_values().plot.bar(ax=axes[1,1], edgecolor='black', color='green')
axes[1,1].set_title('SLA Compliance by Warehouse')
axes[1,1].tick_params(axis='x', rotation=0)
axes[1,1].axhline(0.90, color='red', linestyle='--', alpha=0.5, label='90% target')
axes[1,1].legend()

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Stage Breakdown by Warehouse

In [ ]:
wh_stages = df.groupby('Warehouse')[stages].mean().round(1)
print('Avg Stage Duration by Warehouse (min):')
print(wh_stages)

fig, ax = plt.subplots(figsize=(12, 6))
wh_stages.plot.bar(stacked=True, ax=ax, edgecolor='black')
ax.set_title('Fulfillment Stage Composition by Warehouse')
ax.set_ylabel('Minutes')
ax.legend(title='Stage', bbox_to_anchor=(1.05, 1))
ax.tick_params(axis='x', rotation=0)
ax.axhline(sla_target, color='red', linestyle='--', label=f'SLA: {sla_target}min')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'warehouse_stages.png'), dpi=100, bbox_inches='tight')
plt.show()

## Stage Breakdown by Order Type

In [ ]:
type_stages = df.groupby('OrderType')[stages].mean().round(1)
print('Avg Stage Duration by Order Type (min):')
print(type_stages)

fig, ax = plt.subplots(figsize=(12, 6))
type_stages.plot.bar(stacked=True, ax=ax, edgecolor='black')
ax.set_title('Fulfillment Stage Composition by Order Type')
ax.set_ylabel('Minutes')
ax.legend(title='Stage', bbox_to_anchor=(1.05, 1))
ax.tick_params(axis='x', rotation=0)
ax.axhline(sla_target, color='red', linestyle='--')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'type_stages.png'), dpi=100, bbox_inches='tight')
plt.show()

## Bottleneck Identification

In [ ]:
# Which stage is the longest for each order?
df['Bottleneck'] = df[stages].idxmax(axis=1)
bottleneck_dist = df['Bottleneck'].value_counts()
print('Most common bottleneck stage:')
print(bottleneck_dist)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bottleneck_dist.plot.bar(ax=axes[0], edgecolor='black', color='coral')
axes[0].set_title('Overall Bottleneck Frequency')
axes[0].tick_params(axis='x', rotation=45)

bn_wh = pd.crosstab(df['Warehouse'], df['Bottleneck'], normalize='index').round(3)
bn_wh.plot.bar(stacked=True, ax=axes[1], edgecolor='black')
axes[1].set_title('Bottleneck Distribution by Warehouse')
axes[1].legend(title='Stage', bbox_to_anchor=(1.05, 1))
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'bottleneck.png'), dpi=100, bbox_inches='tight')
plt.show()

## Monthly Trend

In [ ]:
monthly = df.groupby('YearMonth').agg(
    avg_total=('TotalMinutes', 'mean'),
    sla_rate=('OnTimeSLA', 'mean'),
).round(2)

fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.plot(range(len(monthly)), monthly['avg_total'], 'b-o', label='Avg Total (min)')
ax1.set_ylabel('Minutes', color='blue')
ax1.set_xticks(range(0, len(monthly), 3))
ax1.set_xticklabels(monthly.index[::3].astype(str), rotation=45)
ax2 = ax1.twinx()
ax2.plot(range(len(monthly)), monthly['sla_rate'], 'r-s', label='SLA Rate')
ax2.set_ylabel('SLA Compliance', color='red')
ax1.set_title('Monthly Fulfillment Performance')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'monthly_trend.png'), dpi=100, bbox_inches='tight')
plt.show()

## Warehouse × Order Type Heatmap

In [ ]:
cross = df.groupby(['Warehouse', 'OrderType'])['TotalMinutes'].mean().unstack().round(0)
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(cross, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax)
ax.set_title('Avg Fulfillment Time (min): Warehouse × Order Type')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'heatmap.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **Picking** is the most common bottleneck — especially at WH-West and WH-South, suggesting layout or staffing issues
- **Packing** becomes the bottleneck for Bulk and Fragile orders — specialized packing stations could help
- **WH-West** has the lowest SLA compliance — it has the slowest picking AND packing times
- **Bulk orders** exceed the SLA on average — consider a separate Bulk SLA or dedicated bulk lanes
- **QC** is relatively fast but highly variable — outlier QC times disproportionately impact total fulfillment

## Limitations
- Synthetic data with simplified stage logic
- No labor cost or headcount data
- No parallel processing (stages are sequential)
- No order complexity weighting (item count, SKU variety)
- No rework or return-to-stage loops

## How to Improve This Project
- Use real WMS data with actual timestamps per stage
- Add labor utilization and cost-per-order analysis
- Include order complexity (line items, weight, volume)
- Build simulation models for capacity planning
- Add real-time bottleneck detection dashboards

## Production Considerations
- Real-time stage-level tracking dashboards
- Automated alerts when any stage exceeds threshold
- Shift-level performance tracking for workforce management
- Capacity simulation before peak periods

## Common Mistakes
- Looking only at total time without stage decomposition
- Comparing warehouses without accounting for order type mix
- Setting uniform SLAs regardless of order complexity
- Optimizing the wrong stage (non-bottleneck improvement wastes resources)

## Mini Challenge / Exercises
1. What % of SLA breaches are caused by picking alone exceeding 60 minutes?
2. If WH-West's picking time matched WH-North's, what would its SLA compliance become?
3. Which order type has the most consistent (lowest variance) total fulfillment time?
4. Calculate the 90th percentile fulfillment time for each warehouse. How does it compare to the SLA?

## Final Summary / Key Takeaways
- Order fulfillment bottleneck analysis reveals which specific stage to optimize for maximum impact
- Bottlenecks vary by warehouse and order type — one-size-fits-all solutions miss the mark
- Stage-level stacked charts and bottleneck frequency analysis are powerful diagnostic tools
- SLA compliance must account for order complexity to be fair and actionable
- Continuous stage-level monitoring enables proactive operations management